## 1. Import Libraries

Library yang digunakan:
- **pandas** untuk membaca dan mengolah data tabular.
- **numpy** untuk pengolahan nilai numerik dan missing value.
- **re** untuk membersihkan dan membaca pola teks.
- **display** untuk menampilkan DataFrame di Google Colab.
- **google.colab.drive** untuk mengakses dataset yang disimpan di Google Drive.

In [1]:
import pandas as pd
import numpy as np
import re
from IPython.display import display
from google.colab import drive

# Pengaturan tampilan DataFrame
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.max_rows", 100)

## 2. Membaca Dataset Kotor Transaksi Laundry Sepatu

Dataset yang digunakan adalah dataset transaksi Laundry sepatu yang masih mengandung variasi penulisan, missing value, format angka/tanggal yang berbeda, dan ID order yang muncul lebih dari satu kali.

In [2]:
# Mount Google Drive
drive.mount("/content/drive")

# Lokasi dataset transaksi cuci sepatu
file_path = "/content/drive/MyDrive/Datmin2418023/Data Kotor Laundry Sepatu.csv"

# Membaca dataset
df = pd.read_csv(file_path)

# Menampilkan dataset awal dalam bentuk tabel
display(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id_order,nama_pelanggan,tanggal_order,jenis_sepatu,bahan_sepatu,tingkat_kekotoran,jenis_layanan,lama_pengerjaan,biaya,kasir
0,CS012,Audrey,17/05/2026,sneakers,Suede,berat,Deep Clean,express (3 jam),127000,NaN
1,CS037,Fania,20 bulan 5 2026,sneakers,-,Ringan,repaint,13 jam,121000,Mba Ayu
2,CS002,Maya,18 bulan 6 2026,sneakers,Kanvas,RINGAN,cuci reguler,NaN,36000,Mba Ayu
3,CS021,Nessa,2026-04-19,NaN,Kulit,RINGAN,cuci reguler,NaN,34000,Bang Bagus
4,CS011,Nessa,NaN,Sneakers!!,kanvas,Ringan,unyellowing,express (3 jam),27000,NaN
5,CS026,Fania**,NaN,Sneakers,Suede,Berat,deep clean,1hari,105000,Kak Rian
6,CS031,Fania,04/04/2026,SNEAKERS,Kanvas,RINGAN,unyellowing,1 hari,37000,NaN
7,CS016,Rakhis,04/22/2026,Sepatu Kulit,-,Berat,Deep Clean,1hari,109000,Bang Dedi
8,CS041,Fania,03/09/2026,Sepatu Sekolah,Kulit,Ringan,Repaint,NaN,-50000,Bang Dedi
9,CS009,Maya,24/05/2026,NaN,Kulit,Sedang,deep clean,3 hari,109000,Kak Rian


## 3. Analisis Kondisi Awal Dataset

Tahap ini dilakukan sebelum cleansing untuk mengetahui kondisi awal dataset.

Yang diperiksa:
- jumlah baris dan kolom,
- jumlah missing value,
- jumlah duplikat baris secara penuh,
- jumlah duplikat berdasarkan **ID order**.

In [3]:
# Analisis Kondisi dataset awal
analisis_awal = pd.DataFrame({
    "Keterangan": [
        "Jumlah Baris",
        "Jumlah Kolom",
        "Jumlah Missing Value",
        "Duplikat Baris Penuh",
        "Duplikat ID Order",
        "Jumlah ID Order Unik"
    ],
    "Hasil": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        int(df["id_order"].duplicated().sum()),
        int(df["id_order"].nunique())
    ]
})

display(analisis_awal)

,Keterangan,Hasil
0,Jumlah Baris,50
1,Jumlah Kolom,10
2,Jumlah Missing Value,52
3,Duplikat Baris Penuh,0
4,Duplikat ID Order,2
5,Jumlah ID Order Unik,48


## 4. Struktur Variabel Dataset

Menampilkan tipe data, jumlah missing value, dan jumlah nilai unik pada setiap kolom. Tahap ini membantu menentukan jenis cleansing yang diperlukan pada masing-masing variabel.

In [4]:
# Struktur dataset
struktur = pd.DataFrame({
    "Nama Kolom": df.columns,
    "Tipe Data": df.dtypes.astype(str).values,
    "Jumlah Missing": df.isna().sum().values,
    "Jumlah Nilai Unik": df.nunique(dropna=True).values
})

display(struktur)

,Nama Kolom,Tipe Data,Jumlah Missing,Jumlah Nilai Unik
0,id_order,object,0,48
1,nama_pelanggan,object,0,16
2,tanggal_order,object,6,42
3,jenis_sepatu,object,8,9
4,bahan_sepatu,object,3,6
5,tingkat_kekotoran,object,3,6
6,jenis_layanan,object,0,9
7,lama_pengerjaan,object,13,11
8,biaya,object,2,43
9,kasir,object,17,4


## 5. Data Cleansing - Standarisasi Kolom ID Order

Tujuan: membuat penulisan ID transaksi konsisten.

Perubahan:
- spasi awal/akhir dihapus,
- huruf diseragamkan menjadi uppercase.

Contoh:
` cs012 ` → `CS012`

In [5]:
df["id_order"] = (
    df["id_order"]
    .astype("string")
    .str.strip()
    .str.upper()
)

display(df[["id_order"]])

,id_order
0,CS012
1,CS037
2,CS002
3,CS021
4,CS011
5,CS026
6,CS031
7,CS016
8,CS041
9,CS009


## 6. Data Cleansing - Standarisasi Kolom Nama Pelanggan

Tujuan: membersihkan nama pelanggan agar tidak memiliki simbol yang tidak diperlukan dan format kapitalisasi menjadi konsisten.

Perubahan:
- spasi awal/akhir dihapus,
- karakter selain huruf dan spasi dihapus,
- spasi ganda dirapikan,
- kapitalisasi dibuat title case.

Contoh:
`Fania**` → `Fania`

In [6]:
df["nama_pelanggan"] = (
    df["nama_pelanggan"]
    .astype("string")
    .str.strip()
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

display(df[["id_order", "nama_pelanggan"]])

,id_order,nama_pelanggan
0,CS012,Audrey
1,CS037,Fania
2,CS002,Maya
3,CS021,Nessa
4,CS011,Nessa
5,CS026,Fania
6,CS031,Fania
7,CS016,Rakhis
8,CS041,Fania
9,CS009,Maya


## 7. Data Cleansing - Standarisasi Kolom Tanggal Order

Dataset menggunakan beberapa bentuk tanggal, misalnya:
- `2026-04-19`
- `17/05/2026`
- `05/23/2026`
- `20 bulan 5 2026`

Jadi kolom tanggal harus dicleansing terlebih dahulu

In [7]:
def parse_tanggal(x):
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT

    s = re.sub(r"\s+", " ", str(x).strip().lower())

    # Format: 20 bulan 5 2026
    m = re.match(r"(\d{1,2})\s*bulan\s*(\d{1,2})\s*(\d{4})$", s)
    if m:
        hari, bulan, tahun = map(int, m.groups())
        try:
            return pd.Timestamp(tahun, bulan, hari)
        except ValueError:
            return pd.NaT

    # Format ISO: 2026-04-19 atau 2026/04/19
    if re.match(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}$", s):
        return pd.to_datetime(s, errors="coerce")

    # Format slash: DD/MM/YYYY atau MM/DD/YYYY
    m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", s)
    if m:
        angka1, angka2, tahun = map(int, m.groups())

        if angka2 > 12:
            bulan, hari = angka1, angka2
        elif angka1 > 12:
            hari, bulan = angka1, angka2
        else:
            hari, bulan = angka1, angka2

        try:
            return pd.Timestamp(tahun, bulan, hari)
        except ValueError:
            return pd.NaT

    return pd.NaT


df["tanggal_order"] = df["tanggal_order"].apply(parse_tanggal)
df["tanggal_order"] = pd.to_datetime(df["tanggal_order"], errors="coerce")

display(df[["id_order", "tanggal_order"]])

,id_order,tanggal_order
0,CS012,2026-05-17
1,CS037,2026-05-20
2,CS002,2026-06-18
3,CS021,2026-04-19
4,CS011,NaT
5,CS026,NaT
6,CS031,2026-04-04
7,CS016,2026-04-22
8,CS041,2026-09-03
9,CS009,2026-05-24


## 8. Data Cleansing - Standarisasi Kolom Jenis Sepatu

Tujuan: menyatukan variasi penulisan jenis sepatu menjadi kategori yang konsisten.

Contoh:
- `SNEAKERS` → `Sneakers`
- `Sneakers!!` → `Sneakers`

In [8]:
df["jenis_sepatu"] = (
    df["jenis_sepatu"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

peta_sepatu = {
    "sneakers": "Sneakers",
    "sepatu kulit": "Sepatu Kulit",
    "sepatu sekolah": "Sepatu Sekolah",
    "boots": "Boots",
    "sepatu futsal": "Sepatu Futsal",
    "sepatu kanvas": "Sepatu Kanvas"
}

df["jenis_sepatu"] = df["jenis_sepatu"].replace(peta_sepatu)

display(df[["id_order", "jenis_sepatu"]])

,id_order,jenis_sepatu
0,CS012,Sneakers
1,CS037,Sneakers
2,CS002,Sneakers
3,CS021,<NA>
4,CS011,Sneakers
5,CS026,Sneakers
6,CS031,Sneakers
7,CS016,Sepatu Kulit
8,CS041,Sepatu Sekolah
9,CS009,<NA>


## 9. Data Cleansing - Standarisasi Kolom Bahan Sepatu

Tujuan: menyeragamkan penulisan bahan dan mengubah tanda `-` atau teks kosong menjadi missing value.

Contoh:
- `kanvas` → `Kanvas`
- `kulit` → `Kulit`
- `-` → missing

In [9]:
df["bahan_sepatu"] = (
    df["bahan_sepatu"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["bahan_sepatu"] = df["bahan_sepatu"].replace({
    "-": pd.NA,
    "": pd.NA,
    "nan": pd.NA
})

peta_bahan = {
    "kanvas": "Kanvas",
    "kulit": "Kulit",
    "suede": "Suede",
    "mesh": "Mesh"
}

df["bahan_sepatu"] = df["bahan_sepatu"].replace(peta_bahan)

display(df[["id_order", "bahan_sepatu"]])

,id_order,bahan_sepatu
0,CS012,Suede
1,CS037,<NA>
2,CS002,Kanvas
3,CS021,Kulit
4,CS011,Kanvas
5,CS026,Suede
6,CS031,Kanvas
7,CS016,<NA>
8,CS041,Kulit
9,CS009,Kulit


## 10. Data Cleansing - Standarisasi Kolom Tingkat Kekotoran

Tujuan: menyamakan kapitalisasi kategori:
- `ringan` → `Ringan`
- `sedang` → `Sedang`
- `berat` → `Berat`

In [10]:
df["tingkat_kekotoran"] = (
    df["tingkat_kekotoran"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["tingkat_kekotoran"] = df["tingkat_kekotoran"].replace({
    "ringan": "Ringan",
    "sedang": "Sedang",
    "berat": "Berat"
})

display(df[["id_order", "tingkat_kekotoran"]])

,id_order,tingkat_kekotoran
0,CS012,Berat
1,CS037,Ringan
2,CS002,Ringan
3,CS021,Ringan
4,CS011,Ringan
5,CS026,Berat
6,CS031,Ringan
7,CS016,Berat
8,CS041,Ringan
9,CS009,Sedang


## 11. Data Cleansing - Standarisasi Jenis Layanan

Tujuan: menyeragamkan kategori layanan.

Contoh:
- `deep clean` → `Deep Clean`
- `repaint` → `Repaint`
- `cuci reguler` → `Cuci Reguler`
- `unyellowing` → `Unyellowing`

In [11]:
df["jenis_layanan"] = (
    df["jenis_layanan"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["jenis_layanan"] = df["jenis_layanan"].replace({
    "deep clean": "Deep Clean",
    "repaint": "Repaint",
    "cuci reguler": "Cuci Reguler",
    "unyellowing": "Unyellowing"
})

display(df[["id_order", "jenis_layanan"]])

,id_order,jenis_layanan
0,CS012,Deep Clean
1,CS037,Repaint
2,CS002,Cuci Reguler
3,CS021,Cuci Reguler
4,CS011,Unyellowing
5,CS026,Deep Clean
6,CS031,Unyellowing
7,CS016,Deep Clean
8,CS041,Repaint
9,CS009,Deep Clean


## 12. Data Cleansing - Standardisasi Lama Pengerjaan

Semua lama pengerjaan diubah menjadi **jam**.

Contoh:
- `13 jam` → `13`
- `1 hari` → `24`
- `2 hari` → `48`
- `express (3 jam)` → `3`
- `-5 jam` → missing karena nilai negatif tidak valid.

Nilai numerik ini disimpan pada kolom `lama_pengerjaan`.

In [12]:
# STANDARDISASI LAMA PENGERJAAN

def parse_durasi(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    # Format dalam jam
    m = re.search(r"(-?\d+(?:[.,]\d+)?)\s*jam", s)

    if m:
        nilai = float(m.group(1).replace(",", "."))

        # Nilai jam negatif dianggap tidak valid
        if nilai < 0:
            return np.nan

        if nilai.is_integer():
            return f"{int(nilai)} jam"

        return f"{nilai:g} jam"

    # Format dalam hari, dikonversi menjadi jam
    m = re.search(r"(-?\d+(?:[.,]\d+)?)\s*hari", s)

    if m:
        nilai = float(m.group(1).replace(",", "."))

        # Nilai hari negatif dianggap tidak valid
        if nilai < 0:
            return np.nan

        jam = nilai * 24

        if jam.is_integer():
            return f"{int(jam)} jam"

        return f"{jam:g} jam"

    # Format yang tidak dikenali dianggap missing
    return np.nan


# Terapkan standardisasi
df["lama_pengerjaan"] = df["lama_pengerjaan"].apply(parse_durasi)


# Tampilkan hasil standardisasi
display(
    df[["id_order", "lama_pengerjaan"]]
)

,id_order,lama_pengerjaan
0,CS012,3 jam
1,CS037,13 jam
2,CS002,NaN
3,CS021,NaN
4,CS011,3 jam
5,CS026,24 jam
6,CS031,24 jam
7,CS016,24 jam
8,CS041,NaN
9,CS009,72 jam


## 13. Data Cleansing - Standarisasi Biaya

Semua biaya diubah menjadi angka rupiah.

Contoh:
- `Rp 139rb` → `139000`
- `Rp 82rb` → `82000`
- `44k` → `44000`
- `gratis` → `0`
- `-50000` → missing karena biaya negatif dianggap tidak valid.

In [13]:
def parse_biaya(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower().replace(" ", "")

    if s == "gratis":
        return 0.0

    s = s.replace("rp", "")

    if s.endswith("rb"):
        angka = s[:-2].replace(",", ".")
        try:
            return float(angka) * 1000
        except ValueError:
            return np.nan

    if s.endswith("jt"):
        angka = s[:-2].replace(",", ".")
        try:
            return float(angka) * 1_000_000
        except ValueError:
            return np.nan

    if s.endswith("k"):
        angka = s[:-1].replace(",", ".")
        try:
            return float(angka) * 1000
        except ValueError:
            return np.nan

    s = s.replace(".", "").replace(",", "")

    try:
        return float(s)
    except ValueError:
        return np.nan


# Bersihkan kolom biaya hanya jika kolom biaya masih ada
if "biaya" in df.columns:
    df["biaya_rp"] = df["biaya"].apply(parse_biaya)
    df.drop(columns=["biaya"], inplace=True)

# Jika biaya_rp sudah ada, tidak perlu dibuat ulang

# Biaya negatif dianggap tidak valid
df.loc[df["biaya_rp"] < 0, "biaya_rp"] = np.nan

display(df[["id_order", "biaya_rp"]])

,id_order,biaya_rp
0,CS012,127000.0
1,CS037,121000.0
2,CS002,36000.0
3,CS021,34000.0
4,CS011,27000.0
5,CS026,105000.0
6,CS031,37000.0
7,CS016,109000.0
8,CS041,NaN
9,CS009,109000.0


## 14. Data Cleansing - Standarisasi Kasir

Membersihkan spasi dan mengubah string kosong menjadi missing value.

In [14]:
df["kasir"] = (
    df["kasir"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["kasir"] = df["kasir"].replace({
    "": pd.NA,
    "nan": pd.NA
})

display(df[["id_order", "kasir"]])

,id_order,kasir
0,CS012,<NA>
1,CS037,Mba Ayu
2,CS002,Mba Ayu
3,CS021,Bang Bagus
4,CS011,<NA>
5,CS026,Kak Rian
6,CS031,<NA>
7,CS016,Bang Dedi
8,CS041,Bang Dedi
9,CS009,Kak Rian


## 15. Missing Sebelum Imputasi

Mengecek jumlah missing value pada dataset sebelum dilakukan proses imputasi.

In [15]:
# =====================================================
# CEK MISSING VALUE SEBELUM IMPUTASI
# =====================================================

# Menyamakan beberapa bentuk nilai kosong menjadi NaN
nilai_missing = ["", " ", "nan", "NaN", "None", "null"]

df = df.replace(nilai_missing, np.nan)


# Membuat tabel jumlah missing value setiap kolom
missing_sebelum = pd.DataFrame({
    "Nama Kolom": df.columns,
    "Jumlah Missing": df.isna().sum().values
})


# Menampilkan hasil
display(missing_sebelum)

,Nama Kolom,Jumlah Missing
0,id_order,0
1,nama_pelanggan,0
2,tanggal_order,6
3,jenis_sepatu,8
4,bahan_sepatu,12
5,tingkat_kekotoran,3
6,jenis_layanan,0
7,lama_pengerjaan,14
8,kasir,17
9,biaya_rp,3


## 15. Pengisian Missing Value

### Bahan sepatu
Missing bahan diisi berdasarkan **aturan kontekstual jenis sepatu**. Jika masih kosong, digunakan modus.

### Kolom kategorikal
`jenis_sepatu`, `tingkat_kekotoran`, `jenis_layanan`, dan `kasir` menggunakan modus.

### Kolom numerik
`lama_pengerjaan_jam` dan `biaya_rp` menggunakan median.

### Tanggal
Missing `tanggal_order` diisi menggunakan median tanggal.

In [18]:
# PENANGANAN DAN IMPUTASI MISSING VALUE
# 1. BAHAN SEPATU
# Imputasi berdasarkan jenis sepatu
aturan_bahan = {
    "Sepatu Kulit": "Kulit",
    "Sepatu Kanvas": "Kanvas",
    "Boots": "Kulit",
    "Sepatu Futsal": "Mesh"
}

for jenis, bahan in aturan_bahan.items():
    kondisi = (
        (df["jenis_sepatu"] == jenis) &
        (df["bahan_sepatu"].isna())
    )

    df.loc[kondisi, "bahan_sepatu"] = bahan


# Jika masih ada missing pada bahan sepatu,
# gunakan nilai yang paling sering muncul (modus)
if df["bahan_sepatu"].isna().sum() > 0:
    df["bahan_sepatu"] = df["bahan_sepatu"].fillna(
        df["bahan_sepatu"].mode()[0]
    )

# 2. KOLOM KATEGORIKAL
# Imputasi menggunakan modus
kolom_modus = [
    "jenis_sepatu",
    "tingkat_kekotoran",
    "jenis_layanan",
    "kasir"
]

for kolom in kolom_modus:
    if df[kolom].isna().sum() > 0:
        df[kolom] = df[kolom].fillna(
            df[kolom].mode()[0]
        )


# Imputasi menggunakan modus

if df["lama_pengerjaan"].isna().sum() > 0:
    df["lama_pengerjaan"] = df["lama_pengerjaan"].fillna(
        df["lama_pengerjaan"].mode()[0]
    )

# 4. BIAYA
# Imputasi menggunakan median

if df["biaya_rp"].isna().sum() > 0:
    df["biaya_rp"] = df["biaya_rp"].fillna(
        df["biaya_rp"].median()
    )

# 5. TANGGAL ORDER
# Imputasi menggunakan median tanggal

df["tanggal_order"] = pd.to_datetime(
    df["tanggal_order"],
    errors="coerce"
)

if df["tanggal_order"].isna().sum() > 0:

    median_tanggal = df["tanggal_order"].dropna().median()

    df["tanggal_order"] = df["tanggal_order"].fillna(
        median_tanggal
    )

# 6. CEK HASIL IMPUTASI

missing_setelah = pd.DataFrame({
    "Nama Kolom": df.columns,
    "Jumlah Missing": df.isna().sum().values
})

display(missing_setelah)

,Nama Kolom,Jumlah Missing
0,id_order,0
1,nama_pelanggan,0
2,tanggal_order,0
3,jenis_sepatu,0
4,bahan_sepatu,0
5,tingkat_kekotoran,0
6,jenis_layanan,0
7,lama_pengerjaan,0
8,kasir,0
9,biaya_rp,0


## 16. Audit dan Deduplicasi ID Order

Sebelum menghapus data, ID order yang muncul lebih dari satu kali diperiksa terlebih dahulu.

Hal ini berbeda dari pemeriksaan duplikat baris penuh. Pada dataset ini terdapat ID order yang berulang dengan isi transaksi yang tidak sepenuhnya sama.

> **ID order diperlakukan sebagai identitas unik transaksi dan kemunculan pertama dipertahankan (`keep="first"`).**

Dengan audit sebelum penghapusan, proses deduplikasi menjadi transparan.

In [19]:
# Audit ID order sebelum deduplikasi
duplikat_id = (
    df[df["id_order"].duplicated(keep=False)]
    .sort_values("id_order")
    .reset_index(drop=True)
)

display(duplikat_id)

# Deduplikasi berdasarkan ID order
jumlah_sebelum = len(df)

df = df.drop_duplicates(
    subset=["id_order"],
    keep="first"
).copy()

jumlah_sesudah = len(df)

hasil_deduplikasi = pd.DataFrame({
    "Keterangan": [
        "Jumlah data sebelum deduplikasi",
        "Jumlah data sesudah deduplikasi",
        "Jumlah data yang dihapus"
    ],
    "Jumlah": [
        jumlah_sebelum,
        jumlah_sesudah,
        jumlah_sebelum - jumlah_sesudah
    ]
})

display(hasil_deduplikasi)

,id_order,nama_pelanggan,tanggal_order,jenis_sepatu,bahan_sepatu,tingkat_kekotoran,jenis_layanan,lama_pengerjaan,kasir,biaya_rp
0,CS008,Dea,2026-05-23,Boots,Mesh,Ringan,Repaint,24 jam,Bang Dedi,120000.0
1,CS008,Audrey,2026-02-03,Boots,Mesh,Ringan,Repaint,24 jam,Bang Dedi,120000.0
2,CS023,Moren,2026-05-06,Sneakers,Suede,Ringan,Deep Clean,24 jam,Mba Ayu,139000.0
3,CS023,Dani,2026-03-22,Sneakers,Suede,Ringan,Deep Clean,24 jam,Mba Ayu,139000.0


,Keterangan,Jumlah
0,Jumlah data sebelum deduplikasi,50
1,Jumlah data sesudah deduplikasi,48
2,Jumlah data yang dihapus,2


## 17. Data Enrichment

Setelah cleansing inti selesai, dibuat beberapa informasi tambahan dari data yang sudah ada:

1. `bulan_order` — nama bulan berdasarkan tanggal order.
2. `kategori_biaya` — kategori biaya berdasarkan batas nominal yang digunakan dalam notebook.
3. `estimasi_lama_hari` — hasil konversi lama pengerjaan dari jam ke hari.

Data enrichment bukan pembersihan data inti, tetapi penambahan atribut agar dataset lebih informatif untuk analisis.

In [20]:
# DATA ENRICHMENT
# 1. TANGGAL ORDER

df["tanggal_order"] = pd.to_datetime(
    df["tanggal_order"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# 2. BULAN ORDER
# Mengambil bulan dari tanggal order

df["bulan_order"] = pd.to_datetime(
    df["tanggal_order"],
    errors="coerce"
).dt.month_name()

# 3. KATEGORI BIAYA

def kategori_biaya(biaya):
    if biaya < 50000:
        return "Rendah"
    elif biaya <= 100000:
        return "Sedang"
    else:
        return "Tinggi"


df["kategori_biaya"] = df["biaya_rp"].apply(
    kategori_biaya
)

# 4. ESTIMASI LAMA PENGERJAAN DALAM HARI

angka_jam = (
    df["lama_pengerjaan"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

df["estimasi_lama_hari"] = (
    angka_jam / 24
).round(2)

# 5. TAMPILKAN HASIL ENRICHMENT

display(
    df[
        [
            "id_order",
            "tanggal_order",
            "lama_pengerjaan",
            "biaya_rp",
            "bulan_order",
            "kategori_biaya",
            "estimasi_lama_hari"
        ]
    ]
)

,id_order,tanggal_order,lama_pengerjaan,biaya_rp,bulan_order,kategori_biaya,estimasi_lama_hari
0,CS012,2026-05-17,3 jam,127000.0,May,Tinggi,0.12
1,CS037,2026-05-20,13 jam,121000.0,May,Tinggi,0.54
2,CS002,2026-06-18,24 jam,36000.0,June,Rendah,1.00
3,CS021,2026-04-19,24 jam,34000.0,April,Rendah,1.00
4,CS011,2026-04-12,3 jam,27000.0,April,Rendah,0.12
5,CS026,2026-04-12,24 jam,105000.0,April,Tinggi,1.00
6,CS031,2026-04-04,24 jam,37000.0,April,Rendah,1.00
7,CS016,2026-04-22,24 jam,109000.0,April,Tinggi,1.00
8,CS041,2026-09-03,24 jam,102000.0,September,Tinggi,1.00
9,CS009,2026-05-24,72 jam,109000.0,May,Tinggi,3.00


## 19. Validasi Dataset Bersih

Validasi dilakukan setelah seluruh proses untuk memastikan:
- jumlah baris dan kolom sesuai hasil proses,
- tidak ada missing value,
- tidak ada duplikat ID order,
- jumlah ID unik sama dengan jumlah baris.

In [21]:
validasi = pd.DataFrame({
    "Keterangan": [
        "Jumlah Baris",
        "Jumlah Kolom",
        "Total Missing Value",
        "Jumlah Duplikat ID Order",
        "Jumlah ID Order Unik"
    ],
    "Hasil": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df["id_order"].duplicated().sum()),
        int(df["id_order"].nunique())
    ]
})

display(validasi)

assert df.isna().sum().sum() == 0, "Masih terdapat missing value."
assert df["id_order"].duplicated().sum() == 0, "Masih terdapat duplikat ID order."
assert df["id_order"].nunique() == len(df), "Jumlah ID unik tidak sama dengan jumlah baris."

,Keterangan,Hasil
0,Jumlah Baris,48
1,Jumlah Kolom,13
2,Total Missing Value,0
3,Jumlah Duplikat ID Order,0
4,Jumlah ID Order Unik,48


## 20. Data Bersih — Hasil Akhir

Berikut adalah dataset yang telah melewati seluruh proses cleansing, deduplikasi, enrichment, dan validasi.

In [23]:
print("=" * 100)
print("DATASET HASIL CLEANSING")
print("=" * 100)

display(df)

DATASET HASIL CLEANSING


,id_order,nama_pelanggan,tanggal_order,jenis_sepatu,bahan_sepatu,tingkat_kekotoran,jenis_layanan,lama_pengerjaan,kasir,biaya_rp,bulan_order,kategori_biaya,estimasi_lama_hari
0,CS012,Audrey,2026-05-17,Sneakers,Suede,Berat,Deep Clean,3 jam,Bang Bagus,127000.0,May,Tinggi,0.12
1,CS037,Fania,2026-05-20,Sneakers,Kanvas,Ringan,Repaint,13 jam,Mba Ayu,121000.0,May,Tinggi,0.54
2,CS002,Maya,2026-06-18,Sneakers,Kanvas,Ringan,Cuci Reguler,24 jam,Mba Ayu,36000.0,June,Rendah,1.00
3,CS021,Nessa,2026-04-19,Sneakers,Kulit,Ringan,Cuci Reguler,24 jam,Bang Bagus,34000.0,April,Rendah,1.00
4,CS011,Nessa,2026-04-12,Sneakers,Kanvas,Ringan,Unyellowing,3 jam,Bang Bagus,27000.0,April,Rendah,0.12
5,CS026,Fania,2026-04-12,Sneakers,Suede,Berat,Deep Clean,24 jam,Kak Rian,105000.0,April,Tinggi,1.00
6,CS031,Fania,2026-04-04,Sneakers,Kanvas,Ringan,Unyellowing,24 jam,Bang Bagus,37000.0,April,Rendah,1.00
7,CS016,Rakhis,2026-04-22,Sepatu Kulit,Kulit,Berat,Deep Clean,24 jam,Bang Dedi,109000.0,April,Tinggi,1.00
8,CS041,Fania,2026-09-03,Sepatu Sekolah,Kulit,Ringan,Repaint,24 jam,Bang Dedi,102000.0,September,Tinggi,1.00
9,CS009,Maya,2026-05-24,Sneakers,Kulit,Sedang,Deep Clean,72 jam,Kak Rian,109000.0,May,Tinggi,3.00


## 21. Kesimpulan

**Proses data cleansing berhasil memperbaiki kualitas dataset transaksi laundry sepatu melalui standardisasi format, penanganan missing value, audit dan deduplikasi ID order, serta penambahan informasi melalui data enrichment. Setelah proses dilakukan, dataset menjadi lebih konsisten, terstruktur, dan siap digunakan untuk tahap analisis selanjutnya.**